In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from enum import Enum
from dataclasses import dataclass
import math
from tqdm import tqdm
from torch import nn
import torchvision
import torch.nn.functional as F

from tiles import *
from utils import *
from solver import FactorioSim

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE = 'cpu'

In [ ]:
class UNET(nn.Module):
    def __init__(self,chann_in,chann_out):
        super(UNET, self).__init__()

        def dub(cin,cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout,3,1,1), nn.BatchNorm2d(cout), nn.GELU(),
                nn.Conv2d(cout,cout,3,1,1), nn.BatchNorm2d(cout), nn.GELU()
            )
        
        self.d1 = dub(chann_in, 16)
        self.d2 = dub(16,32)
        self.d3 = dub(32,64)

        self.b = dub(64,128)

        self.u3 = dub(128+64,64)
        self.u2 = dub(64+32,32)
        self.u1 = dub(32+16,16)
        
        self.ff = nn.Sequential(
            nn.Conv2d(16, 16,1,1,1), nn.BatchNorm2d(16), nn.GELU(),
            nn.Conv2d(16,chann_out,1,1,1),
        )

        self.p = nn.MaxPool2d(2,2)
    
    def forward(self,x):
        d1 = self.d1(x)
        d2 = self.d2(self.p(d1))
        d3 = self.d3(self.p(d2))

        b = self.b(self.p(d3))

        u3 = self.u3(torch.cat([d3, F.interpolate(b,scale_factor=2)],dim=1))
        u2 = self.u2(torch.cat([d2, F.interpolate(u3,scale_factor=2)],dim=1))
        u1 = self.u1(torch.cat([d1, F.interpolate(u2,scale_factor=2)],dim=1))

        ff = self.ff(u1)

        return ff[:, :, 2:-2,2:-2]


In [ ]:
num_tile = len(Tile)  # 2 for startpos and endpos
box_size = 32
sim = FactorioSim(box_size=32,tile_data=DEFAULT_TILE_DATA)
mdl = UNET(num_tile+2,num_tile).to(DEVICE)

opt = torch.optim.AdamW(mdl.parameters(), lr=0.001)

In [ ]:
def custom_loss(y_P,start_tensor,end_pos):
    print(y_P.size(),start_tensor.size())
    items = sim.eval(y_P,iter_add=start_tensor,max_iter=box_size*1.5)
    return -( items[end_pos[0],end_pos[1]].max(dim=-1)[0] - np.log(100) )

for _ in tqdm(range(100)):
    start_pos = torch.randint(0,box_size, (2,))
    end_pos = torch.randint(0,box_size, (2,))

    start_tensor = torch.zeros(box_size,box_size,device=DEVICE)
    start_tensor[start_pos] = 100

    se_tensor = torch.zeros(box_size,box_size,2,device=DEVICE)
    se_tensor[start_pos][0] = 1 
    se_tensor[end_pos][1] = 1 

    x_logits = torch.randn((box_size,box_size,len(Tile)),device=DEVICE)

    x = torch.cat([x_logits,se_tensor],dim=-1)

    y_logits = mdl(x.permute(2,0,1).unsqueeze(0)).squeeze(0).permute(1,2,0)

    y_P = torch.nn.functional.log_softmax(y_logits,dim=-1)
    loss = custom_loss(y_P,start_tensor,end_pos)

    print(loss.item())
    loss.backward()
    opt.step()
    opt.zero_grad()